# 01 — BlueSky Moderation Data Collection
Cross-Platform Content Moderation Dataset**

Collects labeled posts from BlueSky using the AT Protocol public API and stores them in a shared SQLite database.

### APIs used
| Endpoint | Purpose |
|---|---|
| `app.bsky.feed.searchPosts` | Search posts by keyword; labels embedded in response |
| `app.bsky.labeler.getServices` | Fetch labeler metadata and label definitions |
| `app.bsky.actor.getProfiles` | Fetch profile bio for account-level labels |



---
## 1. Imports & Configuration

In [19]:
import sqlite3
import requests
import time
import logging
from datetime import datetime, timezone
from pathlib import Path

#  Global Variables/Configuraion
DB_PATH       = "../data/moderation.db"
API_BASE      = "https://api.bsky.app/xrpc"
REQUEST_DELAY = 1.0    
LIMIT         = 100    
MAX_PAGES     = 10  

# Moderation discussion queries — posts about moderation topics (low label rate)
SEARCH_QUERIES = [
    "hate speech",
    "harassment",
    "misinformation",
    "spam",
    "threatening",
]

# Label-targeted queries — content actually labeled by the moderation service
# These yield labeled posts since they search for the content being moderated
LABEL_QUERIES = [
    "porn",
    "nudity",
    "graphic-media",
]

ALL_QUERIES = SEARCH_QUERIES + LABEL_QUERIES

# Known public labeler DIDs
# Source: https://docs.bsky.app/docs/advanced-guides/api-directory
LABELER_DIDS = [
    "did:plc:ar7c4by46qjdydhdevvrndac",  # Bluesky Moderation Service (official)
]

# Header value — tells the AppView which labelers to include in responses
# NOTE: cursor-based pagination (page > 1) returns 403 on the public API without auth.
# We use date-windowed pagination via the 'until' parameter instead.
LABELERS_HEADER = ",".join(LABELER_DIDS)

HEADERS = {
    "Accept": "application/json",
    "User-Agent": "dsci511-moderation-research/1.0",
    "atproto-accept-labelers": LABELERS_HEADER,
}

# Ensure data directory exists
Path(DB_PATH).parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("bluesky")

print("✓ Config loaded")
print(f"  DB path          : {DB_PATH}")
print(f"  API base         : {API_BASE}")
print(f"  Labelers         : {len(LABELER_DIDS)}")
print(f"  Discussion queries: {SEARCH_QUERIES}")
print(f"  Label queries    : {LABEL_QUERIES}")

✓ Config loaded
  DB path          : ./data/moderation.db
  API base         : https://api.bsky.app/xrpc
  Labelers         : 1
  Discussion queries: ['hate speech', 'harassment', 'misinformation', 'spam', 'threatening']
  Label queries    : ['porn', 'nudity', 'graphic-media']


---
## 2. Connectivity Check
Verify the API is reachable before doing anything else.

In [20]:
resp = requests.get(
    f"{API_BASE}/app.bsky.feed.searchPosts",
    params={"q": "test", "limit": 1},
    headers=HEADERS,
    timeout=10
)

if resp.status_code == 200:
    posts = resp.json().get("posts", [])
    if posts:
        print("✓ API reachable — status 200")
        print(f"  Sample post text: {posts[0]['record']['text'][:80]}")
    else:
        print("✓ API reachable — status 200 (no posts returned for test query, that's OK)")
else:
    print(f"✗ API returned {resp.status_code}")
    print(f"  Body: {resp.text[:300]}")
    print("  → Make sure you are using https://api.bsky.app (NOT public.api.bsky.app)")

✓ API reachable — status 200
  Sample post text: FREE Samples of Perfume
Luxury fragrance lovers, don’t skip this. Snag a free sa


---
## 3. Database Setup

Four tables — all with `UNIQUE` constraints to prevent duplicate rows:

| Table | Primary key / Unique constraint |
|---|---|
| `labelers` | `did` (PRIMARY KEY) |
| `label_definitions` | `UNIQUE(labeler_did, identifier)` |
| `bsky_posts` | `uri` (PRIMARY KEY) |
| `bsky_labels` | `UNIQUE(uri, label_val, label_src)` |

In [21]:
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.row_factory = sqlite3.Row
    return conn


def setup_db():
    conn = get_conn()
    cur  = conn.cursor()

    cur.executescript("""
        CREATE TABLE IF NOT EXISTS labelers (
            did          TEXT PRIMARY KEY,
            name         TEXT,
            created_at   TEXT,
            fetched_at   TEXT
        );

        CREATE TABLE IF NOT EXISTS label_definitions (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            labeler_did     TEXT NOT NULL REFERENCES labelers(did),
            identifier      TEXT NOT NULL,
            severity        TEXT,
            blurs           TEXT,
            default_setting TEXT,
            description     TEXT,
            UNIQUE(labeler_did, identifier)
        );

        CREATE TABLE IF NOT EXISTS bsky_posts (
            uri             TEXT PRIMARY KEY,
            cid             TEXT,
            author_did      TEXT,
            author_handle   TEXT,
            text            TEXT,
            lang            TEXT,
            post_created_at TEXT,
            like_count      INTEGER DEFAULT 0,
            reply_count     INTEGER DEFAULT 0,
            repost_count    INTEGER DEFAULT 0,
            search_query    TEXT,
            fetched_at      TEXT
        );

        CREATE TABLE IF NOT EXISTS bsky_labels (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            uri         TEXT NOT NULL,
            cid         TEXT,
            label_val   TEXT NOT NULL,
            label_src   TEXT NOT NULL,
            is_negation INTEGER DEFAULT 0,
            labeled_at  TEXT,
            platform    TEXT DEFAULT 'bluesky',
            UNIQUE(uri, label_val, label_src)
        );

        CREATE INDEX IF NOT EXISTS idx_bsky_labels_uri ON bsky_labels(uri);
        CREATE INDEX IF NOT EXISTS idx_bsky_labels_val ON bsky_labels(label_val);
        CREATE INDEX IF NOT EXISTS idx_bsky_posts_uri  ON bsky_posts(uri);
    """)

    # Migration: add search_query column if created by an older schema
    existing_posts_cols = {row[1] for row in cur.execute("PRAGMA table_info(bsky_posts)")}
    if "search_query" not in existing_posts_cols:
        cur.execute("ALTER TABLE bsky_posts ADD COLUMN search_query TEXT")
        print("  ↳ migrated bsky_posts: added search_query column")

    # Migration: drop and recreate bsky_labels if it has a FK on label_src
    # (older schema used REFERENCES labelers(did) which blocks labels from
    #  any labeler not already in our labelers table)
    fk_list = cur.execute("PRAGMA foreign_key_list(bsky_labels)").fetchall()
    if any(row[2] == "labelers" for row in fk_list):
        # Safe to recreate since bsky_labels was always written after bsky_posts
        row_count = cur.execute("SELECT COUNT(*) FROM bsky_labels").fetchone()[0]
        cur.executescript("""
            DROP TABLE IF EXISTS bsky_labels;
            CREATE TABLE bsky_labels (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                uri         TEXT NOT NULL,
                cid         TEXT,
                label_val   TEXT NOT NULL,
                label_src   TEXT NOT NULL,
                is_negation INTEGER DEFAULT 0,
                labeled_at  TEXT,
                platform    TEXT DEFAULT 'bluesky',
                UNIQUE(uri, label_val, label_src)
            );
            CREATE INDEX IF NOT EXISTS idx_bsky_labels_uri ON bsky_labels(uri);
            CREATE INDEX IF NOT EXISTS idx_bsky_labels_val ON bsky_labels(label_val);
        """)
        print(f"  ↳ migrated bsky_labels: removed FK constraint on label_src "
              f"(dropped {row_count} rows — will be re-collected)")

    conn.commit()
    conn.close()
    print("✓ Database schema ready")


setup_db()

✓ Database schema ready


---
## 4. Helper Functions

In [22]:
def api_get(endpoint, params=None):
    """GET request to BlueSky API with error handling and rate limiting."""
    url = f"{API_BASE}/{endpoint}"
    try:
        resp = requests.get(url, params=params, headers=HEADERS, timeout=15)
        if resp.status_code == 429:
            log.warning("Rate limited — waiting 30s")
            time.sleep(30)
            resp = requests.get(url, params=params, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return resp.json()
    except requests.HTTPError as e:
        log.warning(f"HTTP {resp.status_code} — {endpoint}: {resp.text[:200]}")
        return None
    except requests.RequestException as e:
        log.error(f"Request failed — {endpoint}: {e}")
        return None


def now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


print("✓ Helpers defined")

✓ Helpers defined


---
## 5. Collect Labeler Metadata

Fetches each labeler's name and their label definitions (what `hate`, `spam`, etc. means to that labeler — severity, blurs, description). This is the comparison data that makes cross-platform analysis possible.

In [23]:
def fetch_labeler_metadata():
    data = api_get("app.bsky.labeler.getServices", params={
        "dids": LABELER_DIDS,
        "detailed": "true"
    })

    if not data or "views" not in data:
        print("✗ No labeler data returned")
        print(f"  Raw response: {data}")
        return

    conn = get_conn()
    cur  = conn.cursor()
    labelers_new = 0
    defs_new     = 0

    for view in data["views"]:
        creator  = view.get("creator", {})
        did      = creator.get("did", "")
        name     = creator.get("displayName") or creator.get("handle", "unknown")
        created  = view.get("indexedAt", "")
        policies = view.get("policies", {})

        cur.execute("""
            INSERT OR IGNORE INTO labelers (did, name, created_at, fetched_at)
            VALUES (?, ?, ?, ?)
        """, (did, name, created, now_iso()))
        labelers_new += cur.rowcount

        for defn in policies.get("labelValueDefinitions", []):
            identifier  = defn.get("identifier", "")
            severity    = defn.get("severity", "")
            blurs       = defn.get("blurs", "")
            default_s   = defn.get("defaultSetting", "")
            description = ""
            for locale in defn.get("locales", []):
                if locale.get("lang") == "en":
                    description = locale.get("description", "")
                    break
            if not description and defn.get("locales"):
                description = defn["locales"][0].get("description", "")

            cur.execute("""
                INSERT OR IGNORE INTO label_definitions
                    (labeler_did, identifier, severity, blurs, default_setting, description)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (did, identifier, severity, blurs, default_s, description))
            defs_new += cur.rowcount

        print(f"  Labeler : {name} ({did[:30]}...)")
        print(f"  Labels defined: {list(policies.get('labelValues', []))}")

    conn.commit()
    conn.close()
    print(f"\n✓ Labelers stored    : {labelers_new} new")
    print(f"  Label definitions  : {defs_new} new")


fetch_labeler_metadata()

  Labeler : Bluesky Moderation Service (did:plc:ar7c4by46qjdydhdevvrnd...)
  Labels defined: ['!hide', '!warn', 'porn', 'sexual', 'nudity', 'sexual-figurative', 'graphic-media', 'self-harm', 'sensitive', 'extremist', 'intolerant', 'threat', 'rude', 'illicit', 'security', 'unsafe-link', 'impersonation', 'misinformation', 'scam', 'engagement-farming', 'spam', 'rumor', 'misleading', 'inauthentic']

✓ Labelers stored    : 0 new
  Label definitions  : 0 new


---
## 6. Collect Posts + Embedded Labels

Searches for posts by keyword. Labels are returned **embedded** in each post object under `post.labels[]` because we pass the `atproto-accept-labelers` header. Both the post and its labels are stored together.

**Deduplication:** `bsky_posts` has `uri` as PRIMARY KEY, and `bsky_labels` has `UNIQUE(uri, label_val, label_src)`. Re-running skips anything already stored.

In [24]:
import re as _re

# Valid AT Protocol label values: lowercase letters, digits, hyphens, optional leading '!'
# Rejects metadata tags like comment_count:23 or reaction_count:142
_VALID_LABEL = _re.compile(r'^!?[a-z][a-z0-9-]*$')


def store_post_and_labels(cur, post, query):
    """Store a single post and its embedded labels. Returns (post_new, labels_new)."""
    uri    = post.get("uri", "")
    cid    = post.get("cid", "")
    author = post.get("author", {})
    record = post.get("record", {})
    langs  = record.get("langs", [])

    cur.execute("""
        INSERT OR IGNORE INTO bsky_posts
            (uri, cid, author_did, author_handle, text, lang,
             post_created_at, like_count, reply_count, repost_count,
             search_query, fetched_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        uri, cid,
        author.get("did", ""), author.get("handle", ""),
        record.get("text", ""),
        langs[0] if langs else None,
        record.get("createdAt", ""),
        post.get("likeCount", 0), post.get("replyCount", 0), post.get("repostCount", 0),
        query, now_iso()
    ))
    post_new = cur.rowcount

    labels_new = 0
    for label in post.get("labels", []):
        label_val  = label.get("val", "")
        label_src  = label.get("src", "")
        is_neg     = 1 if label.get("neg", False) else 0
        labeled_at = label.get("cts", "")

        if not label_val or not label_src:
            continue
        # Skip non-moderation metadata labels (e.g. comment_count:23, reaction_count:142)
        if not _VALID_LABEL.match(label_val):
            continue

        cur.execute("""
            INSERT OR IGNORE INTO bsky_labels
                (uri, cid, label_val, label_src, is_negation, labeled_at)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (uri, cid, label_val, label_src, is_neg, labeled_at))
        labels_new += cur.rowcount

    return post_new, labels_new


def collect_posts_for_query(query):
    """Paginate through searchPosts using date windowing (cursor pagination
    is blocked after page 1 on the unauthenticated public API)."""
    from datetime import timedelta

    until_ts     = None
    page         = 0
    total_posts  = 0
    total_labels = 0

    print(f"\n  Query: '{query}'")

    while page < MAX_PAGES:
        params = {"q": query, "limit": LIMIT}
        if until_ts:
            params["until"] = until_ts

        data = api_get("app.bsky.feed.searchPosts", params=params)

        if not data or "posts" not in data or not data["posts"]:
            break

        posts  = data["posts"]
        conn   = get_conn()
        cur_db = conn.cursor()

        for post in posts:
            p, l = store_post_and_labels(cur_db, post, query)
            total_posts  += p
            total_labels += l

        conn.commit()
        conn.close()

        page += 1
        print(f"    Page {page:>2} — fetched {len(posts):>3} | "
              f"new posts: {total_posts} | new labels: {total_labels}", end="\r")

        # Advance the window: subtract 1s from the oldest post's indexedAt
        oldest = min(p.get("indexedAt", "") for p in posts if p.get("indexedAt"))
        if not oldest:
            break
        try:
            dt = datetime.fromisoformat(oldest.replace("Z", "+00:00"))
            until_ts = (dt - timedelta(seconds=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
        except ValueError:
            break

    print(f"    Done — {total_posts} new posts, {total_labels} new labels stored")
    return total_posts, total_labels


def collect_all_posts():
    grand_posts  = 0
    grand_labels = 0
    for query in ALL_QUERIES:
        p, l = collect_posts_for_query(query)
        grand_posts  += p
        grand_labels += l
    print(f"\n{'─'*50}")
    print(f"Total new posts stored  : {grand_posts}")
    print(f"Total new labels stored : {grand_labels}")


collect_all_posts()


  Query: 'hate speech'
    Done — 0 new posts, 0 new labels storednew labels: 0

  Query: 'harassment'
    Done — 3 new posts, 0 new labels storednew labels: 0

  Query: 'misinformation'
    Done — 6 new posts, 0 new labels storednew labels: 0

  Query: 'spam'
    Done — 8 new posts, 0 new labels storednew labels: 0

  Query: 'threatening'
    Done — 10 new posts, 0 new labels storednew labels: 0

  Query: 'porn'
    Done — 42 new posts, 26 new labels storedew labels: 26

  Query: 'nudity'
    Done — 1 new posts, 0 new labels storednew labels: 0

  Query: 'graphic-media'
    Done — 0 new posts, 0 new labels storednew labels: 0

──────────────────────────────────────────────────
Total new posts stored  : 70
Total new labels stored : 26


---
## 7. Verify — Row Counts

In [25]:
import pandas as pd

conn = get_conn()

print("── Table sizes ──────────────────────────────────")
for t in ["labelers", "label_definitions", "bsky_posts", "bsky_labels"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22}: {n:>6} rows")

conn.close()

── Table sizes ──────────────────────────────────
  labelers              :      3 rows
  label_definitions     :     58 rows
  bsky_posts            :   7792 rows
  bsky_labels           :   1491 rows


---
## 8. Verify — Label Distribution

In [26]:
conn = get_conn()

df_dist = pd.read_sql_query("""
    SELECT
        l.label_val,
        COUNT(*)                                          AS count,
        COUNT(DISTINCT l.label_src)                       AS from_n_labelers,
        COALESCE(
            MAX(CASE WHEN lb.did IS NOT NULL THEN lb.name END),
            'community labeler'
        )                                                 AS top_labeler
    FROM bsky_labels l
    LEFT JOIN labelers lb ON l.label_src = lb.did
    GROUP BY l.label_val
    ORDER BY count DESC
""", conn)

conn.close()
print("── Label distribution ───────────────────────────────────────────")
print(df_dist.to_string(index=False))

── Label distribution ───────────────────────────────────────────
    label_val  count  from_n_labelers        top_labeler
         porn   1095              142 Bluesky Moderation
       sexual    231               19 Bluesky Moderation
       nudity    102               68 Bluesky Moderation
graphic-media     62               60 Bluesky Moderation
    self-harm      1                1 Bluesky Moderation


---
## 9. Verify — Sample Joined Records

In [27]:
conn = get_conn()

# severity/blurs are available only for custom-defined labels (labelValueDefinitions).
# Built-in AT Protocol labels (porn, sexual, nudity, graphic-media, !warn, !hide)
# are not returned in labelValueDefinitions — their defaults are baked into AT clients.
# Custom labels (self-harm, misinformation, extremist, etc.) DO have definitions stored.

df_sample = pd.read_sql_query("""
    SELECT
        SUBSTR(p.text, 1, 80)                  AS text_preview,
        p.search_query,
        l.label_val,
        COALESCE(lb.name, l.label_src)         AS labeler,
        COALESCE(ld.severity, '(built-in)')    AS severity,
        COALESCE(ld.blurs,    '(built-in)')    AS blurs,
        l.labeled_at
    FROM bsky_labels l
    JOIN bsky_posts p       ON l.uri       = p.uri
    LEFT JOIN labelers lb   ON l.label_src = lb.did
    LEFT JOIN label_definitions ld
           ON ld.labeler_did = l.label_src
          AND ld.identifier  = l.label_val
    WHERE p.text IS NOT NULL AND p.text != ''
      AND lb.name IS NOT NULL          -- only official/named labelers
    ORDER BY (ld.severity IS NOT NULL) DESC,  -- rows with definitions first
             l.labeled_at DESC
    LIMIT 5
""", conn)

conn.close()
pd.set_option("display.max_colwidth", 85)
df_sample

,text_preview,search_query,label_val,labeler,severity,blurs,labeled_at
0,okay why my body kinda pretty what 😥 love my legs bro cuts are the cherry on top,graphic-media,self-harm,Bluesky Moderation,alert,content,2026-05-14T04:24:33.635Z
1,Aiden Borg takes a ride on Jaxon Coltons big cock🍆🔥\r\n\r\n👉 JOIN THE WORLD’S FIRST,porn,porn,Bluesky Moderation,(built-in),(built-in),2026-05-20T20:33:57.474Z
2,💋 https://Fucky.Mom 👈Live Sex Party veronicaredonda 🍌TikTok Girls Cams🍌 CamMode,porn,porn,Bluesky Moderation,(built-in),(built-in),2026-05-20T20:33:47.410Z
3,💖💖💖\n\n#porn #adult #gooning #xxx #adult #18+ #nude #bound #fishnet #onlyfans #sex,porn,porn,Bluesky Moderation,(built-in),(built-in),2026-05-20T20:31:36.339Z
4,➡️ Full video available: https://futanari.it.com/video/1230/bx9\n\n🔥 Ai Hentai Por,porn,porn,Bluesky Moderation,(built-in),(built-in),2026-05-20T20:30:43.429Z


---
## 10. Verify — Posts Without Labels
Posts that matched our search query but had no labels applied — useful to understand label sparsity.

In [28]:
conn = get_conn()

total    = conn.execute("SELECT COUNT(*) FROM bsky_posts").fetchone()[0]
labeled  = conn.execute("""
    SELECT COUNT(DISTINCT uri) FROM bsky_labels
""").fetchone()[0]
unlabeled = total - labeled

conn.close()
print(f"Total posts collected : {total}")
print(f"Posts with labels     : {labeled}")
print(f"Posts without labels  : {unlabeled}")
if total > 0:
    print(f"Label coverage        : {labeled/total*100:.1f}%")
    print()
    print("Note: most posts will NOT have labels — only content that has been")
    print("actively flagged by a labeler shows up with a label value.")
    print("Low coverage is expected and normal.")

Total posts collected : 7792
Posts with labels     : 1198
Posts without labels  : 6594
Label coverage        : 15.4%

Note: most posts will NOT have labels — only content that has been
actively flagged by a labeler shows up with a label value.
Low coverage is expected and normal.


---
## Summary

**What was collected:**
- Labeler metadata and label definitions from configured labeler DIDs
- Posts matching each search query (English only, `lang=en`)
- Labels embedded in post responses via `atproto-accept-labelers` header

**Deduplication:**
- `bsky_posts.uri` — PRIMARY KEY, one row per post
- `bsky_labels` — UNIQUE(uri, label_val, label_src), one row per label event
- All inserts use `INSERT OR IGNORE` — re-running is safe

**Known limitation:**
Label coverage will be low — most posts do not have moderation labels. This is expected. The labeled posts are the rare, valuable signal.

**Next:** `02_lemmy_collection.ipynb` — collect moderation logs from Lemmy instances into the same `moderation.db`.